In [ ]:
import pandas as pd

In [ ]:
cnes = pd.read_csv('arquivos/CNES/cnes_estabelecimentos.csv', sep=';', decimal=',', encoding='latin1', low_memory=False)

In [ ]:
cnes.info()

In [ ]:
import os
import glob

# Pasta contendo os arquivos CSV
pasta_sim = "arquivos/SIM"

# Lista todos os arquivos CSV, ordenados
arquivos = sorted(glob.glob(os.path.join(pasta_sim, "dados_*.csv")))

# Lista para armazenar os DataFrames filtrados
dfs_filtrados = []

for arquivo in arquivos:
    print(f"Processando: {arquivo}")
    
    # Lê o CSV com separador ponto-e-vírgula e encoding latin1
    df = pd.read_csv(arquivo, sep=";", encoding="latin1", dtype=str)
    
    # Normaliza os nomes das colunas para MAIÚSCULO
    df.columns = df.columns.str.upper()
    
    # Verifica se a coluna TPMORTEOCO existe
    if "TPMORTEOCO" not in df.columns:
        print(f"  ERRO: coluna TPMORTEOCO não encontrada em {arquivo}")
        continue
    
    # Filtra registros com TPMORTEOCO em {1, 2, 3, 4}
    df_filtrado = df[df["TPMORTEOCO"].isin(["1", "2", "3", "4"])]
    
    print(f"  Total registros: {len(df)} | Registros filtrados: {len(df_filtrado)}")
    
    if len(df_filtrado) > 0:
        dfs_filtrados.append(df_filtrado)

# Concatena todos os DataFrames filtrados
if dfs_filtrados:
    df_final = pd.concat(dfs_filtrados, ignore_index=True)
    print(f"\nTotal de registros filtrados (todos os anos): {len(df_final)}")
    
    # Salva o resultado
    # df_final.to_csv("registros_mm.csv", sep=";", index=False, encoding="utf-8-sig")
    print("Arquivo registros_mm.csv salvo com sucesso!")
    print(f"Colunas: {list(df_final.columns)}")
else:
    print("Nenhum registro encontrado para os filtros aplicados.")

In [ ]:
for coluna in df_final.columns:
    if 'ESTAB' in coluna:
        print(f"Coluna encontrada: {coluna}")

In [ ]:
# Verifica se os códigos de estabelecimento de df_final existem no CNES
codigo_estab = df_final['CODESTAB'].astype(str).str.strip()
codigos_cnes = cnes['CO_CNES'].astype(str).str.strip()

mask_existencia = codigo_estab.isin(codigos_cnes)

df_final['codigo_existe_no_cnes'] = mask_existencia

print(f"Total de registros em df_final: {len(df_final)}")
print(f"Registros com código presente no CNES: {mask_existencia.sum()}")
print(f"Registros sem correspondência: {(~mask_existencia).sum()}")

# Opcional: mostrar os códigos sem correspondência
codigos_sem_correspondencia = codigo_estab[~mask_existencia].dropna().unique()
print("Exemplos de códigos sem correspondência:")
print(codigos_sem_correspondencia[:10])

In [ ]:
# Verificação tratanto CODESTAB e CO_CNES como valores numéricos
codigo_estab_num = pd.to_numeric(df_final['CODESTAB'], errors='coerce')
codigos_cnes_num = pd.to_numeric(cnes['CO_CNES'], errors='coerce')

mask_existencia_num = codigo_estab_num.isin(codigos_cnes_num.dropna())

df_final['codigo_existe_no_cnes_num'] = mask_existencia_num

print(f"Total de registros em df_final: {len(df_final)}")
print(f"Registros com código presente no CNES (numérico): {mask_existencia_num.sum()}")
print(f"Registros sem correspondência (numérico): {(~mask_existencia_num).sum()}")

In [ ]:
# Cria chaves numéricas para o join
# Garante que as colunas tenham o mesmo tipo para o merge

df_final_join = df_final.copy()
cnes_join = cnes.copy()

df_final_join['CODESTAB_num'] = pd.to_numeric(df_final_join['CODESTAB'], errors='coerce')
cnes_join['CO_CNES_num'] = pd.to_numeric(cnes_join['CO_CNES'], errors='coerce')

# Remove colunas duplicadas/irrelevantes do CNES para evitar conflitos no merge
cnes_join = cnes_join.drop(columns=['CO_CNES'], errors='ignore')

# Left join
merged_df = (
    df_final_join
    .merge(
        cnes_join,
        left_on='CODESTAB_num',
        right_on='CO_CNES_num',
        how='left',
        suffixes=('_df_final', '_cnes')
    )
)

# Remove a chave auxiliar após o merge
merged_df = merged_df.drop(columns=['CODESTAB_num', 'CO_CNES_num'], errors='ignore')

print(f"Shape do df_final: {df_final.shape}")
print(f"Shape do merged_df: {merged_df.shape}")
print("Colunas adicionadas do CNES:")
print([col for col in merged_df.columns if col.endswith('_cnes') or col in cnes.columns])

In [ ]:
merged_df.to_excel("registros_mm_com_cnes.xlsx", index=False)

In [ ]:
# Conta a quantidade de registros por CODMUNOCOR no dataset unido
contagem_codemunocor = merged_df['CODMUNOCOR'].value_counts()

print(contagem_codemunocor.head())
print(f"\nCODMUNOCOR com maior quantidade de registros: {contagem_codemunocor.idxmax()} ({contagem_codemunocor.max()} registros)")

In [ ]:
# Verifica o comprimento dos valores de CODMUNOCOR no dataset unido
codmun = merged_df['CODMUNOCOR'].astype(str).str.strip()
comprimentos = codmun.str.len()

print('Resumo do comprimento de CODMUNOCOR:')
print(comprimentos.value_counts().sort_index())
print(f"\nQuantidade com 6 dígitos: {(comprimentos == 6).sum()}")
print(f"Quantidade com 7 dígitos: {(comprimentos == 7).sum()}")
print(f"Quantidade com mais de 7 dígitos: {(comprimentos > 7).sum()}")

if (comprimentos == 7).sum() > 0:
    print("\nExemplos com 7 dígitos:")
    print(codmun[comprimentos == 7].dropna().head().tolist())

In [ ]:
# Filtra merged_df para o município de São Paulo (CODMUNOCOR = 355030)
merged_df_sp = merged_df[merged_df['CODMUNOCOR'] == '355030'].copy()

print(f"Quantidade de registros após o filtro: {len(merged_df_sp)}")
print(merged_df_sp.head())

In [ ]:
merged_df_sp.to_csv("registros_mm_sp.csv", sep=";", index=False, encoding="utf-8-sig")